### Modules nécessaires

In [1]:
using LinearAlgebra, SparseArrays
using Test

### Implémentation de Bunch-Kaufman

In [2]:
function bunch_kaufman(A)
    """
    Factorise A = LBL' selon la factorisation de Bunch-Parlett, utilisant la technique du pivotage partiel.
    Entrée : Matrice A carrée, hermitienne et indéfinie
    Sortie : Matrice A contenant à la fois L et B :
                - L sur les éléments hors diagonaux
                - B sur les éléments blocs diagonaux ne faisant pas partie de L
             Matrice P
    """
    alpha = (1 + sqrt(17))/8
    n = size(A, 1)
    T = eltype(A)
    P = Matrix{T}(I, n, n)

    i_diagonal = 1
    while i_diagonal < n - 1
        ### Initialisation de la partie de A traitée
        A_ = view(A, i_diagonal:n, i_diagonal:n)
        n_ = size(A_, 1)

        ### Choix du pivot et pivotage
        w_1 = 0
        r = 0
        for i in 2:n_
            magnitude_a_i1 = abs(A_[i, 1])
            if magnitude_a_i1 > w_1
                w_1 = magnitude_a_i1
                r = i
            end
        end

        magnitude_a_11 = abs(A_[1, 1])
        if magnitude_a_11 >= alpha*w_1  # E de taille 1, P ne permute rien
            # Déterminant de E
            inv_det_E = conj(A_[1, 1])/magnitude_a_11^2

            # Complément de Schur
            for i in 2:n_
                for j in 2:n_
                    A_[i, j] -= inv_det_E*(A_[i, 1] * conj(A_[j, 1]))
                end
            end
            
            # Calcul de L
            A_[2:n_, 1] .*= inv_det_E
            A_[1, 2:n_] .= 0

            # Prochain bloc
            i_diagonal += 1
        else
            w_r = 0
            for i in 1:n_
                if i != r
                    magnitude_a_ir = abs(A_[i, r])
                    if magnitude_a_ir > w_r
                        w_r = magnitude_a_ir
                    end
                end
            end

            if magnitude_a_11*w_r >= alpha*w_1^2  # E de taille 1, P ne permute rien
                # Déterminant de E
                inv_det_E = conj(A_[1, 1])/magnitude_a_11^2

                # Complément de Schur
                for i in 2:n_
                    for j in 2:n_
                        A_[i, j] -= inv_det_E*(A_[i, 1] * conj(A_[j, 1]))
                    end
                end

                # Calcul de L
                A_[2:n_, 1] .*= inv_det_E
                A_[1, 2:n_] .= 0

                # Prochain bloc
                i_diagonal += 1
            else
                magnitude_a_rr = abs(A_[r, r])
                if magnitude_a_rr >= alpha*w_r  # E de taille 1, P permute 1 et r
                    # Permutation des lignes et colonnes
                    A_[:, [1, r]] .= A_[:, [r, 1]]
                    A[[i_diagonal, i_diagonal+r-1], :] .= A[[i_diagonal+r-1, i_diagonal], :]
                    P[[i_diagonal, i_diagonal+r-1], :] .= P[[i_diagonal+r-1, i_diagonal], :]

                    # Déterminant de E
                    inv_det_E = conj(A_[1, 1])/magnitude_a_rr^2

                    # Complément de Schur
                    for i in 2:n_
                        for j in 2:n_
                            A_[i, j] -= inv_det_E*(A_[i, 1] * conj(A_[j, 1]))
                        end
                    end

                    # Calcul de L
                    A_[2:n_, 1] .*= inv_det_E
                    A_[1, 2:n_] .= 0
                    
                    # Prochain bloc
                    i_diagonal += 1
                else  # E de taille 2, P permute 2 et r
                    # Permutation des lignes et colonnes
                    A_[:, [2, r]] .= A_[:, [r, 2]]
                    A[[i_diagonal+1, i_diagonal+r-1], :] .= A[[i_diagonal+r-1, i_diagonal+1], :]
                    P[[i_diagonal+1, i_diagonal+r-1], :] .= P[[i_diagonal+r-1, i_diagonal+1], :]

                    # Déterminant de E
                    det_E = (A_[1, 1]*A_[2, 2] - A_[2, 1]*A_[1, 2])
                    inv_det_E = conj(det_E)/abs(det_E)^2

                    # Complément de Schur
                    e_11, e_22, e_21, e_12 = A_[1, 1], A_[2, 2], A_[2, 1], A_[1, 2]
                    for i in 3:n_
                        for j in 3:n_
                            A_[i, j] -= inv_det_E*((A_[i, 1]*e_22 - A_[i, 2]*e_21)*conj(A_[j, 1]) + (A_[i, 2]*e_11 - A_[i, 1]*e_12)*conj(A_[j, 2]))
                        end
                    end

                    # Calcul de L
                    for i in 3:n_
                        a_i1, a_i2 = A_[i, 1], A_[i, 2]
                        A_[i, 1] = inv_det_E*(a_i1*e_22 - a_i2*e_21)
                        A_[i, 2] = inv_det_E*(a_i2*e_11 - a_i1*e_12)
                    end
                    A_[1, 3:n_] .= 0
                    A_[2, 3:n_] .= 0

                    # Prochain bloc
                    i_diagonal += 2
                end
            end
        end
    end

    return A, P
end

function bunch_kaufman_extract_L_B(A)
    """
    Récupère la matrice L et B à partir de la forme compacte de la factorisation de Bunch-Parlett
    Entrée : Matrice A compacte factorisée selon Bunch-Parlett
    Sortie : Matrice L
             Matrice B
    """
    n = size(A, 1)
    T = eltype(A)

    subdiagonal = zeros(T, n-1)
    diagonal = zeros(T, n)
    superdiagonal = zeros(T, n-1)
    L = LowerTriangular(Matrix{T}(I, n, n))

    i_diagonal = 1
    while i_diagonal < n - 1
        if A[i_diagonal, i_diagonal+1] == 0
            # Matrice B
            subdiagonal[i_diagonal] = 0
            diagonal[i_diagonal] = A[i_diagonal, i_diagonal]
            superdiagonal[i_diagonal] = 0

            # Matrice L
            L[i_diagonal + 1:n, i_diagonal] .= A[i_diagonal + 1:n, i_diagonal]

            # Prochain bloc
            i_diagonal += 1
        else
            # Matrice B
            subdiagonal[i_diagonal] = A[i_diagonal+1, i_diagonal]
            diagonal[i_diagonal] = A[i_diagonal, i_diagonal]
            superdiagonal[i_diagonal]  = A[i_diagonal, i_diagonal+1]

            subdiagonal[i_diagonal+1] =  0
            diagonal[i_diagonal + 1] = A[i_diagonal+1, i_diagonal+1]
            superdiagonal[i_diagonal+1]  = 0

            # Matrice L
            L[i_diagonal + 2:n, i_diagonal] .= A[i_diagonal + 2:n, i_diagonal]
            L[i_diagonal + 2:n, i_diagonal + 1] .= A[i_diagonal + 2:n, i_diagonal + 1]

            # Prochain bloc
            i_diagonal += 2
        end
    end

    if i_diagonal == n - 1
        subdiagonal[i_diagonal] = A[i_diagonal+1, i_diagonal]
        diagonal[i_diagonal], diagonal[i_diagonal+1]  = A[i_diagonal, i_diagonal], A[i_diagonal+1, i_diagonal+1]
        superdiagonal[i_diagonal] = A[i_diagonal, i_diagonal+1]
    else
        diagonal[i_diagonal] = A[i_diagonal, i_diagonal]
    end

    B = Tridiagonal(subdiagonal, diagonal, superdiagonal)

    return L, B
end

bunch_kaufman_extract_L_B (generic function with 1 method)

### Validation avec différents types de données

In [3]:
N = 25
m, n = 50, 25
for i in 1:N
    A = randn(m, n)
    T = eltype(A)
    K_0 = [sparse(Matrix{T}(I, m, m)) A; A' spzeros(n,n)]
    K = copy(K_0)

    K, P = bunch_kaufman(K)
    L, B = bunch_kaufman_extract_L_B(K)
    @test norm(P'*L*B*L'*P - K_0) < 1e-10
end

for i in 1:N
    A = randn(ComplexF64, m, n)
    T = eltype(A)
    K_0 = [sparse(Matrix{T}(I, m, m)) A; A' spzeros(n,n)]
    K = copy(K_0)

    K, P = bunch_kaufman(K)
    L, B = bunch_kaufman_extract_L_B(K)
    @test norm(P'*L*B*L'*P - K_0) < 1e-10
    @test eltype(K) == T && eltype(P) == T && eltype(L) == T && eltype(B) == T
end

for i in 1:N
    A = randn(BigFloat, m, n) + im * randn(BigFloat, m, n)
    T = eltype(A)
    K_0 = [sparse(Matrix{T}(I, m, m)) A; A' spzeros(n,n)]
    K = copy(K_0)

    K, P = bunch_kaufman(K)
    L, B = bunch_kaufman_extract_L_B(K)
    @test norm(P'*L*B*L'*P - K_0) < 1e-10
    @test eltype(K) == T && eltype(P) == T && eltype(L) == T && eltype(B) == T
end


### Notes

In [4]:
# if norm(P'*L*B*L'*P - K_0) > 1e-10
#     display(K_0)
#     display(K)
#     display(P)
#     display(L*B*L')
#     display(P'*L*B*L'*P)
#     display("--------")

#     # Crée ou ouvre un fichier texte pour l'écriture
#     filename = "matrices.txt"  # Nom du fichier texte

#     # Ouvrir le fichier en mode écriture (cela crée le fichier si il n'existe pas)
#     open(filename, "w") do file
#         # Écrire le contenu de la matrice dans le fichier
#         println(file, "K_0:")
#         for row in eachrow(K_0)
#             println(file, row)  # Écrire chaque ligne de la matrice
#         end
        
#         println(file, "\nK:")
#         for row in eachrow(K)
#             println(file, row)
#         end

#         println(file, "\nP:")
#         for row in eachrow(P)
#             println(file, row)
#         end
        
#         println(file, "\nL:")
#         for row in eachrow(L)
#             println(file, row)
#         end

#         println(file, "\nB:")
#         for row in eachrow(B)
#             println(file, row)
#         end
        
#         println(file, "\nP'*L*B*L'*P:")
#         for row in eachrow(P'*L*B*L'*P)
#             println(file, row)
#         end
#     end
# end